In [1]:
import numpy as np
import pandas as pd
import struct

In [2]:
def call_files():

    outputs = []
    for i in range (4, 26):
        file_name = f'C:/Users/famirteimoury/openfast/MyOwnProjects/New 22 Wind Turbine/IEA-22-280-RWT/Simulations/03-11-2026/IEA-22-280-RWT-OnLand_Wind_{i}.outb'
        outputs.append (file_name)

    outputfile = outputs[0]
    return outputfile

In [3]:
def define_variables():
    byteorder = 'little'
    if byteorder == 'little': # integer 16 bit
        var1 = '<i2'
    else:
        var1 = '>i2'
    if byteorder == 'little': # integer 32 bit
        var2 = '<i4'
    else:
        var2 = '>i4'
    if byteorder == 'little': # float 64 bit
        var3 = '<f8'
    else:
        var3 = '>f8'
    if byteorder == 'little': # float 32 bit
        var4 = '<f4'
    else:
        var4 = '>f4'
    return  var1, var2, var3, var4 

In [ ]:
def read_binary_file(filenam, var1, var2, var3, var4):
    inx = 0
    with open (filenam, 'rb') as f:
        wind = f.read()
        fileid_part = wind[0: inx + 2]
        fileid = np.frombuffer (fileid_part, dtype= var1, count=1)
        fileid = fileid[0]
        print (fileid)
        inx += 2
        if fileid == 4:
            len_name_part = wind [inx: inx + 2]
            len_name = np.frombuffer (len_name_part, dtype= var1, count= 1)
            len_name = len_name[0]
            inx += 2
        else:
            len_name = 10        
        
        numchans_part = wind[inx : inx + 4]
        numoutchans = np.frombuffer (numchans_part, dtype= var2, count= 1)
        numoutchans = numoutchans[0]
        inx += 4
        
        nt_part = wind[inx: inx + 4]
        nt = np.frombuffer (nt_part, dtype= var2, count= 1)
        nt = nt[0]
        print (f'nt is {nt}')
        inx +=4

        if fileid == 1:
            time_scl_part = wind[inx: inx + 8]
            time_scl = np.frombuffer (time_incr_part, dtype= var3, count=1)
            time_scl = time_scl[0]
            inx += 8
            time_off_part = wind[inx: inx + 8]
            time_off = np.frombuffer (time_off_part, dtype= var3, count = 1)
            time_off = time_off[0]
            print (f'time off is {time_off}')
            inx += 8
        else:
            time_out1_part = wind [inx: inx + 8]
            time_out1 = np.frombuffer (time_out1_part, dtype= var3, count = 1)
            time_out1 = time_out1[0]
            print (f'time out is {time_out1}')
            inx += 8
            time_incr_part = wind [inx: inx + 8]
            time_incr = np.frombuffer (time_incr_part, dtype= var3, count = 1)
            time_incr = time_incr[0]
            inx += 8
        
        if fileid == 3:
            col_scl = np.ones(numoutchans)
            col_off = np.zeros (numoutchans)
        else:
            col_scl = np.zeros(numoutchans)
            for i in range (numoutchans):
                col_scl_part = wind [inx: inx +4]
                col_scl[i] = np.frombuffer (col_scl_part, dtype= var4, count=1)[0]
                inx += 4
            col_off = np.zeros(numoutchans)
            for i in range (numoutchans):
                col_off_part = wind [inx: inx+4]
                col_off[i] = np.frombuffer (col_off_part, dtype=var4, count=1)[0]
                inx +=4
        
        len_desc = np.frombuffer( wind[inx:inx+4], dtype= var2, count=1)[0]
        inx += 4
        desc_str = wind[inx:inx+len_desc].decode('utf-8').strip()
        inx += len_desc

        channel_name = []
        for i in range (numoutchans +1):
            name_part = wind [inx: inx + len_name]
            channel_name.append (name_part.decode('utf-8').strip())
            inx += len_name
        channel_units = []
        for i in range (numoutchans + 1):
            unit_p = wind[inx: inx+ len_name]
            unit = unit_p.decode('utf-8').strip()
            channel_units.append(unit)
            inx += len_name
        channels = np.zeros((nt, numoutchans + 1))


        if fileid ==1:
            time_integer = wind [inx: inx +4*nt]
            time_pack = np.frombuffer (time_incr_part, dtype= var2, count = nt)
            channels[i, 0] = (time_pack-time_off)/time_scl
            inx += 4*nt
        else:
            channels[:, 0] = time_out1 + time_incr * np.arange (nt)
            
        
        n_pts = nt * numoutchans
        print (f'inx is {inx}')
        print (f'n_pts is {n_pts}')
        print (f'type num out channesl is {type(numoutchans)}')
        if fileid == 3:  # NoCompressWithoutTime
            part = wind [inx: inx+ (8*n_pts)]
            packed_data = np.array(np.frombuffer(part, dtype= var3, count = n_pts))
        else:
            part = wind [inx: inx + 2*n_pts]
            print (type(part))
            packed_data = np.frombuffer(part, dtype=var1, count=n_pts)

        print (f'type of packed data is {type(packed_data)}')
        for it in range (nt):
            start_p = numoutchans*it
            channels[it, 1:] = (packed_data [start_p: start_p + numoutchans] - col_off)/col_scl

    chan_full_name = np.add(channel_name, channel_units)
    df = pd.DataFrame(channels, columns=chan_full_name)   
    print (df)
    return df


        

In [5]:
if __name__ == '__main__':
    fname = call_files()
    var1, var2, var3, var4 = define_variables()
    df = read_binary_file(fname, var1, var2, var3, var4)
    


4
nt is 305001
time out is 0.0
inx is 2213
n_pts is 21350070
type num out channesl is <class 'numpy.int32'>
<class 'bytes'>
type of packed data is <class 'numpy.ndarray'>
        Time(s)  Wind1VelX(m/s)  Wind1VelY(m/s)  Wind1VelZ(m/s)  Azimuth(deg)  \
0         0.000             4.0             0.0             0.0      0.000000   
1         0.002             4.0             0.0             0.0      0.082398   
2         0.004             4.0             0.0             0.0      0.164797   
3         0.006             4.0             0.0             0.0      0.247195   
4         0.008             4.0             0.0             0.0      0.329593   
...         ...             ...             ...             ...           ...   
304996  609.992             4.0             0.0             0.0    189.950153   
304997  609.994             4.0             0.0             0.0    189.977620   
304998  609.996             4.0             0.0             0.0    190.005086   
304999  609.998    